

# DeltaNet / NLP Quick Start 总览

本目录属于 **ascend_tla_deltanet** 样例：在 **昇腾 NPU** 上基于 **DeltaNet（Delta Rule）** 与 **Triton** 完成算子验证与 NLP 训练实践。下列内容帮助你在当前仓库布局下理解背景、目录结构与学习顺序（`00` → `01` → `02`）。

架构总览图等资源见上游开源仓库：[Ascend-TLA](https://gitcode.com/SMULL_Group/Ascend-TLA.git)。


---

## 1. 样例定位与学习目标

本样例聚焦于：

- 在昇腾上用 Triton 实现 **DeltaNet / Delta Rule** 相关算子，并提供可调用路径
- 提供 **PyTorch** 侧参考实现与数值 / 性能验证（见 `01_operator_deltanet.ipynb`）
- 提供 **NLP 语言建模** 的数据、训练与评测流程（见 `02_downstream_nlp.ipynb`）

阅读完成后，你可以：

1. 理解本目录代码组织与 DeltaNet 在本样例中的位置  
2. 跑通 DeltaNet 算子示例并完成前后向一致性检查  
3. 继续进入 NLP 下游训练与评测流程  



---

## 2. 本样例目录结构与学习路径

以下示意 **ascend_tla_deltanet** 与本 Quick Start 的相对位置（路径相对于样例根目录，即与顶层 `README.md` 同级）：

```text
ascend_tla_deltanet/
├── README.md
├── requirements.txt
├── patches/
└── internal/
    ├── quick_start/                    # 当前 Notebook：00 / 01 / 02
    └── tla/
        ├── utils.py
        ├── modules/
        ├── torch/                      # PyTorch 侧参考实现（如 delta_net）
        └── nlp/
            └── tla-nlp-frame/
                ├── preparation/        # 数据下载与分词
                └── training/           # 训练脚本、democonfig、DeltaNet attention 等
                └── tla/                # delta_net相关算子实现
```


建议学习顺序：

1. `00_overview.ipynb`：背景与复杂度动机  
2. `01_operator_deltanet.ipynb`：DeltaNet 算子与一致性 / 性能验证  
3. `02_downstream_nlp.ipynb`：NLP 数据、训练与评测  


---

## 3. 核心公式

### 3.1 Softmax 注意力

给定序列长度为 $N$，注意力定义为：

$$
\mathrm{Attn}(Q,K,V)=\mathrm{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

对第 $i$ 个 token，可写为：

$$
\mathbf{o}_i=\sum_{j=1}^{N}\alpha_{ij}\mathbf{v}_j,\quad
\alpha_{ij}=\frac{\exp\left(\mathbf{q}_i^\top\mathbf{k}_j/\sqrt{d_k}\right)}{\sum_{t=1}^{N}\exp\left(\mathbf{q}_i^\top\mathbf{k}_t/\sqrt{d_k}\right)}
$$

### 3.2 Softmax 注意力的平方复杂度

关键瓶颈在于先构造 $N\times N$ 的相似度矩阵：

$$
S=QK^\top\in\mathbb{R}^{N\times N}
$$

其计算与存储复杂度都随序列长度平方增长：

$$
\text{Time}=\mathcal{O}(N^2d),\quad
\text{Memory}=\mathcal{O}(N^2)
$$

当 $N$ 很大时，$N^2$ 项会成为主要开销。

### 3.3 DeltaNet（Delta Rule）

在上述复杂度动机下，本样例采用 **DeltaNet**：通过带 **$\beta_t$ 门控** 的记忆矩阵秩一更新（递推形式），在保持随序列长度近似线性扩展的可行性的同时，引入相对「只累加记忆」更强的 **覆写** 能力。数学形式、与实现的对应关系及基准测试见 **`01_operator_deltanet.ipynb`**。


---

## 4. 环境准备

本项目默认运行在 **Ascend NPU + CANN** 环境（如云上/服务器 Linux 环境）。下面给出环境配置方法：

### 4.1 最小依赖清单
- **CANN Toolkit + Kernels**：下载页见 [CANN 社区版下载中心](https://www.hiascend.com/developer/download/community/result?module=cann)，安装后执行 `set_env.sh` 使环境变量生效
- **Python/Conda 环境**：建议 Python 3.10
- **PyTorch NPU 适配**：`torch_npu`
- **Triton on Ascend**：参考 [triton-ascend 官方页面](https://ascend.github.io/triton-ascend/)

### 4.2 CANN（Toolkit + Kernels）安装命令（示例）

```bash
# 进入安装包所在目录（示例）
cd /cache

# 安装 toolkit
chmod +x Ascend-cann-toolkit_<version>_linux-<arch>.run
./Ascend-cann-toolkit_<version>_linux-<arch>.run --install

# 安装 kernels
chmod +x Ascend-cann-kernels-<chip_type>_<version>_linux-<arch>.run
./Ascend-cann-kernels-<chip_type>_<version>_linux-<arch>.run --install

# 使环境变量生效
echo 'source ${HOME}/Ascend/ascend-toolkit/set_env.sh' >> ~/.bashrc
source ~/.bashrc
```

### 4.3 Python 与 Triton-Ascend 安装

```bash
# 1) 创建并激活 conda 环境
conda create -n tla python=3.10 -y
conda activate tla

# 2) 安装常用基础依赖
python -m pip install numpy scipy psutil pytest pytest-xdist pyyaml

# 3) 安装 torch_npu（版本需与 CANN 匹配）
python -m pip install torch_npu

# 4) triton-ascend 前置：检查升级 GCC
conda search gcc_linux-aarch64 -c conda-forge
conda install -c conda-forge gcc_linux-aarch64=9.5 gxx_linux-aarch64=9.5 sysroot_linux-aarch64=2.17
cd $CONDA_PREFIX/bin
ln -sf aarch64-conda-linux-gnu-gcc gcc
ln -sf aarch64-conda-linux-gnu-g++ g++
gcc --version

# 5) 安装 triton-ascend
python -m pip install pybind11
python -m pip install triton-ascend
```

### 4.4 可选依赖（NLP）
- **NLP**：`transformers` / `datasets` / `accelerate` / `deepspeed`（用于 `internal/tla/nlp/tla-nlp-frame`）

若 `triton-ascend` 安装仍失败，优先检查 `gcc --version`、`CC`/`CXX` 指向以及 CANN 环境变量是否生效。

下一步请运行上面的环境自检代码，然后打开 `01_operator_deltanet.ipynb`。

---

## 5. 环境自检

完成第 4 节安装配置后，再运行下面代码，确认 Python 包可用、NPU 可见、关键环境变量已生效。

In [4]:
import os
import platform


def _print_kv(k, v):
    print(f"{k:<18}: {v}")


_print_kv("Python", platform.python_version())
_print_kv("System", platform.platform())

try:
    import torch

    _print_kv("torch", getattr(torch, "__version__", "unknown"))
    npu_available = hasattr(torch, "npu") and torch.npu.is_available()
    _print_kv("torch.npu", npu_available)
except Exception as e:
    _print_kv("torch", f"IMPORT FAILED: {e}")

try:
    import torch_npu

    _print_kv("torch_npu", getattr(torch_npu, "__version__", "imported"))
except Exception as e:
    _print_kv("torch_npu", f"IMPORT FAILED: {e}")

try:
    import triton

    _print_kv("triton", getattr(triton, "__version__", "unknown"))
except Exception as e:
    _print_kv("triton", f"IMPORT FAILED: {e}")

# CANN 常见环境变量（不同环境命名可能略有差异）
for k in [
    "ASCEND_HOME_PATH",
    "ASCEND_TOOLKIT_HOME",
    "LD_LIBRARY_PATH",
]:
    v = os.environ.get(k)
    _print_kv(k, "SET" if v else "NOT SET")

Python            : 3.10.19
System            : Linux-4.19.90-vhulk2211.3.0.h1543.eulerosv2r10.aarch64-aarch64-with-glibc2.28
torch             : 2.6.0+cpu
torch.npu         : True
torch_npu         : 2.6.0
triton            : 3.2.0
ASCEND_HOME_PATH  : SET
ASCEND_TOOLKIT_HOME: SET
LD_LIBRARY_PATH   : SET
